### Importation des données

In [ ]:
import sys
sys.path.append('../src')
import utils
import pandas as pd
import s3fs
import geopandas as gpd

Notre première base de données correspond aux arbres dans la ville de Paris. 

In [ ]:
arbres = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/60433484-f30e-44ef-a362-e5553a9b7a42", sep = ";")


In [ ]:
arbres

Notre deuxième base de données correspond aux iris de Paris, triés depuis les iris de la France métropolitaine. 

In [ ]:
fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "raphcrre"
PATH_IRIS = f"{MY_BUCKET}/diffusion/projet_arbres/iris.gpkg"

with fs.open(PATH_IRIS, 'rb') as f:
    iris_france = gpd.read_file(f)

iris = iris_france[iris_france['code_insee'].astype(str).str.startswith('75')].copy()

In [ ]:
iris

### Jointure

Réalisons une jointure entre nos tables pour avoir les informations sur les iris de chaque arbre. 

In [ ]:
df_arbres = utils.jointure_arbres_iris(arbres, iris)


### Nettoyage de la base de données

On remarque que notre table contient trop de colonnes inutiles : 

- IDBASE : id de la ligne inutile
- cleabs : Identifiant technique IGN
- IDEMPLACEMENT, COMPLEMENT ADRESSE, LIEU / ADRESSE : indications géographique inutiles comme on possède la localisation. 
- iris : déjà contenu dans code_iris, qui est un indentifiant complet et unique. 
- geo_point_2d, lat et lon : redondant avec geometry
- 'ARRONDISSEMENT : variable doublée
- index_right : rajoutée avec la jointure
- TYPE EMPLACEMENT : modalité Arbre partout. 

In [ ]:
df_arbres = utils.suppression_colonnes(df_arbres)
df_arbres

In [ ]:
print(df_arbres.columns.tolist())

Suppression des valeurs manquantes et traitement des valeurs aberrantes de circonférence et de hauteur d'arbre. 

In [ ]:
df_arbres = utils.nettoyer_valeurs_aberrantes(df_arbres)

print("-" * 30)
print(f"Statistiques après nettoyage :")
print(f"Hauteur moyenne : {df_arbres['hauteur_m'].mean():.2f} m")
print(f"Circonférence moyenne : {df_arbres['circonference_cm'].mean():.2f} cm")
print("-" * 30)